# Figure 2 — Clustering-method benchmark

## tl;dr

Panel **D** was executed successfully for the declared B004 cohort: 220,082 cells, with 55 Leiden, 300 FlowSOM, 60 SpatialSort, and 50 TIFF-derived PIXIE clusters. The local PNG/PDF and provenance record are written to `outputs/figure_02/`.

## Notebook contract

- **Purpose:** Generate manuscript Figure 2 panels from explicit local inputs.
- **Current scope:** Panel D only.
- **Inputs:** `20251007_cleaned_trainingdata_yang.h5ad`, selected keyed clustering assignments, and TIFF-derived PIXIE assignments.
- **Outputs:** repository-relative `outputs/figure_02/` (local and gitignored).
- **Shared code:** `src/llm_spatial_omics_clustering/figure_02.py`.

## Context & Methods

### Key assumptions

The H5AD-derived cohort contains only the eight B004 `File_ID`s declared in `configs/figure_02.yaml`. The attached Methods text specifies the clustering parameters and the PIXIE TIFF workflow. It does **not** specify UMAP settings or callout bounds; those implementation-derived values remain marked `VERIFY:` in the configuration until author confirmation.

Each method view is colored by the majority ground-truth label within its clusters, so all five UMAPs use the same label palette and shared geometry.

## Data

The loader builds 48 cell-level features directly from the H5AD: its 45 `X` variables plus `obs.CD123`, `obs.Hoechst1`, and `obs.CDX2`. It also validates the exact `(File_ID, ID)` keys, all eight B004 regions, 220,082 cells, nonmissing `cell_type_update` labels, and finite feature values.

## Results

### Panel D — Ground-truth and method-derived UMAP embeddings

In [1]:
# Panel D — Ground Truth and four method-derived UMAP views
# This is the sole executable cell for Figure 2 Panel D. It loads only the
# declared B004 cohort from the source H5AD, then validates each keyed method
# assignment before rendering the five-view panel.
from pathlib import Path
import sys

working_directory = Path.cwd().resolve()
repository_root = next(
    candidate
    for candidate in (working_directory, *working_directory.parents)
    if (candidate / "src" / "llm_spatial_omics_clustering").is_dir()
)
if str(repository_root / "src") not in sys.path:
    sys.path.insert(0, str(repository_root / "src"))

from llm_spatial_omics_clustering.figure_02 import (
    build_panel_d_table,
    build_shared_umap,
    load_b004_h5ad,
    load_color_map,
    load_figure_config,
    load_method_assignments,
    render_panel_d,
    resolve_data_root,
    save_panel_d_provenance,
)

config = load_figure_config(repository_root / "configs" / "figure_02.yaml")
data_root = resolve_data_root(config)
panel_d_data = load_b004_h5ad(config, data_root=data_root)
panel_d_coordinates = build_shared_umap(panel_d_data, config)
panel_d_assignments = load_method_assignments(panel_d_data, config, data_root=data_root)
panel_d_table = build_panel_d_table(
    panel_d_data,
    panel_d_coordinates,
    panel_d_assignments,
    config,
)

output_config = config["panel_d"]["outputs"]
coordinate_path = repository_root / output_config["coordinates_csv"]
coordinate_path.parent.mkdir(parents=True, exist_ok=True)
panel_d_coordinates.to_csv(coordinate_path, index=False)

style = config["panel_d"]["style"]
color_map = load_color_map(
    panel_d_table["Ground Truth"],
    config,
    data_root=data_root,
)
panel_png = render_panel_d(
    panel_d_table,
    repository_root / output_config["panel_png"],
    color_map=color_map,
    zoom=style["zoom"],
    point_size=float(style["point_size"]),
    alpha=float(style["point_alpha"]),
)
panel_pdf = render_panel_d(
    panel_d_table,
    repository_root / output_config["panel_pdf"],
    color_map=color_map,
    zoom=style["zoom"],
    point_size=float(style["point_size"]),
    alpha=float(style["point_alpha"]),
)
provenance_path = save_panel_d_provenance(
    panel_d_data,
    panel_d_table,
    repository_root / output_config["provenance_json"],
    config,
)

print(f"B004 cells: {len(panel_d_table):,}")
print("Clusters:", {name: int(table['cluster'].nunique()) for name, table in panel_d_assignments.items()})
print(f"Coordinates: {coordinate_path}")
print(f"Panel PNG: {panel_png}")
print(f"Panel PDF: {panel_pdf}")
print(f"Provenance: {provenance_path}")


B004 cells: 220,082
Clusters: {'leiden': 55, 'flowsom': 300, 'spatialsort': 60, 'pixie': 50}
Coordinates: /Users/zacharydeutsch/Desktop/cell_masks/LLM-Spatial-omics-Clustering/outputs/figure_02/figure_02d_b004_umap_coordinates.csv
Panel PNG: /Users/zacharydeutsch/Desktop/cell_masks/LLM-Spatial-omics-Clustering/outputs/figure_02/figure_02d_b004_umap_comparison.png
Panel PDF: /Users/zacharydeutsch/Desktop/cell_masks/LLM-Spatial-omics-Clustering/outputs/figure_02/figure_02d_b004_umap_comparison.pdf
Provenance: /Users/zacharydeutsch/Desktop/cell_masks/LLM-Spatial-omics-Clustering/outputs/figure_02/figure_02d_b004_provenance.json


## Figure panels

- [ ] **A:** CODEX feature-extraction and clustering evaluation workflow.
- [ ] **B:** Ground-truth cell counts by cell type.
- [ ] **C:** Ground-truth spatial cell-type map.
- [x] **D:** Ground-truth and method-derived UMAP embeddings — executed for B004.
- [ ] **E:** Weighted F1 scores by clustering method.
- [ ] **F:** F1 scores by cell type and clustering method.
- [ ] **G:** Cluster purity by clustering method.
- [ ] **H:** Cluster purity by cell type and clustering method.
- [ ] **I:** CD8+ T-cell protein-marker expression profiles.
- [ ] **J:** Low- and high-agreement spatial cell-type maps.

## Takeaways

The executed Panel D run covered all 220,082 B004 cells and validated the expected cluster counts: Leiden 55, FlowSOM 300, SpatialSort 60, and TIFF PIXIE 50. The five views use one H5AD-derived UMAP geometry; only the cluster-majority label coloring differs.

## Export

The Panel D cell writes the shared coordinates, PNG, PDF, and provenance JSON to `outputs/figure_02/`. Those generated artifacts stay local by repository policy; the notebook, configuration, source code, and tests are tracked.

## Validation

A successful run must show 220,082 B004 cells and cluster counts of Leiden 55, FlowSOM 300, SpatialSort 60, and TIFF PIXIE 50. The PIXIE manifest must have status `complete` and match the configured TIFF parameters.